<a href="https://colab.research.google.com/github/bashirun-008/prosody_projects/blob/main/prosody_project1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install librosa numpy scikit-learn matplotlib

In [2]:
import librosa
import soundfile as sf
y, sr = librosa.load(librosa.ex('trumpet'))

sf.write("sample_happy.wav", y[:sr*3], sr)       # First 3 sec
sf.write("sample_neutral.wav", y[sr*3:sr*6], sr)   # Middle 3 sec
sf.write("sample_sad.wav", y[sr*6:sr*9], sr)       # Last 3 sec

downloaded_files = ["sample_happy.wav", "sample_neutral.wav", "sample_sad.wav"]
print("Successfully created local audio samples:")
for f in downloaded_files:
    print(" -", f)

Successfully created local audio samples:
 - sample_happy.wav
 - sample_neutral.wav
 - sample_sad.wav


In [10]:
import librosa
import numpy as np


def extract_features(audio_file):
  y, sr = librosa.load(audio_file, sr=None)

  # Check if audio signal is empty or silent
  if len(y) == 0 or np.max(np.abs(y)) == 0:
    return np.zeros(20)  # Return array of zeros for empty/silent audio

  # 1. Volume normalization
  y = y / np.max(np.abs(y))

  # 2. Pitch tracking via PyIN (restricted to human vocal range: 65Hz - 500Hz)
  try:
    f0, voiced_flag, voiced_probs = librosa.pyin(
        y, fmin=65, fmax=500, sr=sr
    )
    pitch_values = f0[~np.isnan(f0)] if f0 is not None else []
  except Exception:
    pitch_values = []

  if len(pitch_values) > 0:
    mean_pitch = np.mean(pitch_values)
    std_pitch = np.std(pitch_values)
    pitch_range = np.ptp(pitch_values)
  else:
    mean_pitch, std_pitch, pitch_range = 0, 0, 0

  # 3. RMS Energy & Spectral Features
  rms = librosa.feature.rms(y=y)[0]
  mean_energy = np.mean(rms)
  std_energy = np.std(rms)

  zcr = (
      np.mean(librosa.feature.zero_crossing_rate(y=y))
      if len(y) > 0
      else 0
  )
  spec_centroid = (
      np.mean(librosa.feature.spectral_centroid(y=y, sr=sr))
      if len(y) > 0
      else 0
  )

  # 4. MFCCs (13 coefficients)
  mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
  mfcc_means = np.mean(mfccs, axis=1)

  return np.hstack([
      mean_pitch,
      std_pitch,
      pitch_range,
      mean_energy,
      std_energy,
      zcr,
      spec_centroid,
      mfcc_means,
  ])


print("Updated extract_features function!")

Updated extract_features function!


In [11]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score


np.random.seed(42)


happy_data = np.random.normal(loc=[350, 200, 0.08], scale=[30, 20, 0.015], size=(100, 3))


neutral_data = np.random.normal(loc=[250, 100, 0.04], scale=[20, 15, 0.010], size=(100, 3))

sad_data = np.random.normal(loc=[150, 50, 0.015], scale=[20, 10, 0.005], size=(100, 3))


X = np.vstack((happy_data, neutral_data, sad_data))
y = np.array(["Happy/Excited"]*100 + ["Neutral/Calm"]*100 + ["Sad/Tired"]*100)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


model = RandomForestClassifier(n_estimators=50, random_state=42)
model.fit(X_train, y_train)

acc = accuracy_score(y_test, model.predict(X_test))
print(f"Model Training Complete!")
print(f"Training Accuracy: {acc * 100:.1f}%")

Model Training Complete!
Training Accuracy: 100.0%


In [12]:
import os
import librosa
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X, y = [], []

# Use the sample files created in Cell 2 to generate synthetic-augmented variations for training
sample_mapping = {
    "sample_happy.wav": "Happy/Excited",
    "sample_neutral.wav": "Neutral/Calm",
    "sample_sad.wav": "Sad/Tired",
}

for sample_file, label in sample_mapping.items():
  if os.path.exists(sample_file):
    # Load base audio
    y_audio, sr = librosa.load(sample_file, sr=None)

    # Generate augmented pitch/speed variations to create a real training set
    for pitch_shift in [-2, -1, 0, 1, 2]:
      for rate in [0.9, 1.0, 1.1]:
        y_mod = librosa.effects.pitch_shift(
            y_audio, sr=sr, n_steps=pitch_shift
        )
        y_mod = librosa.effects.time_stretch(y_mod, rate=rate)

        # Temporary save to extract features
        temp_path = "temp_var.wav"
        import soundfile as sf

        sf.write(temp_path, y_mod, sr)

        try:
          feats = extract_features(temp_path)
          X.append(feats)
          y.append(label)
        except Exception:
          continue

if len(X) > 0:
  X = np.array(X)
  y = np.array(y)

  # Scale features
  scaler = StandardScaler()
  X_scaled = scaler.fit_transform(X)

  # Train/Test Split
  X_train, X_test, y_train, y_test = train_test_split(
      X_scaled, y, test_size=0.2, random_state=42
  )

  # Model Training
  model = RandomForestClassifier(n_estimators=100, random_state=42)
  model.fit(X_train, y_train)

  acc = accuracy_score(y_test, model.predict(X_test))
  print("Model Trained Successfully!")
  print(f"Test Accuracy: {acc * 100:.1f}%")
else:
  print("Error: No audio files found to train on. Make sure Cell 2 was run!")

/usr/local/lib/python3.12/dist-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=0
  warnings.warn(


Model Trained Successfully!
Test Accuracy: 100.0%


In [13]:
# Test predictions on created sample audio files
for file in downloaded_files:
  feats = extract_features(file)

  # Scale features before predicting
  feats_scaled = scaler.transform([feats])

  prediction = model.predict(feats_scaled)[0]
  probs = model.predict_proba(feats_scaled)[0]

  print(f"\nFile: {file}")
  print(f"Predicted Emotion: -> {prediction.upper()} <-")
  for emotion, prob in zip(model.classes_, probs):
    print(f"  - {emotion}: {prob * 100:.1f}%")


File: sample_happy.wav
Predicted Emotion: -> HAPPY/EXCITED <-
  - Happy/Excited: 100.0%
  - Neutral/Calm: 0.0%
  - Sad/Tired: 0.0%

File: sample_neutral.wav
Predicted Emotion: -> NEUTRAL/CALM <-
  - Happy/Excited: 2.0%
  - Neutral/Calm: 98.0%
  - Sad/Tired: 0.0%

File: sample_sad.wav
Predicted Emotion: -> SAD/TIRED <-
  - Happy/Excited: 0.0%
  - Neutral/Calm: 0.0%
  - Sad/Tired: 100.0%


In [14]:
import google.colab.files as files

print("Upload your audio file (.wav, .mp3, or .mpeg):")
uploaded = files.upload()

if len(uploaded) > 0:
  user_file = list(uploaded.keys())[0]

  # Extract & Scale Features
  user_feats = extract_features(user_file)
  user_feats_scaled = scaler.transform([user_feats])

  user_pred = model.predict(user_feats_scaled)[0]
  user_probs = model.predict_proba(user_feats_scaled)[0]

  print("\n==========================================")
  print(f"ANALYSIS FOR: '{user_file}'")
  print("==========================================")
  print(f"Mean Pitch: {user_feats[0]:.1f} Hz")
  print(f"Pitch Range: {user_feats[2]:.1f} Hz")
  print(f"Mean Energy: {user_feats[3]:.4f}")
  print("------------------------------------------")
  print(f"PREDICTED EMOTION: -> {user_pred.upper()} <-")
  print("------------------------------------------")

  print("\nConfidence Scores:")
  for emotion, prob in zip(model.classes_, user_probs):
    print(f"  - {emotion}: {prob * 100:.1f}%")
else:
  print("No file uploaded.")

Upload your audio file (.wav, .mp3, or .mpeg):


Saving my_speech.wav.m4a to my_speech.wav.m4a


/tmp/ipykernel_7962/2523164193.py:6: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(audio_file, sr=None)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)



ANALYSIS FOR: 'my_speech.wav.m4a'
Mean Pitch: 244.9 Hz
Pitch Range: 311.2 Hz
Mean Energy: 0.0737
------------------------------------------
PREDICTED EMOTION: -> NEUTRAL/CALM <-
------------------------------------------

Confidence Scores:
  - Happy/Excited: 35.0%
  - Neutral/Calm: 42.0%
  - Sad/Tired: 23.0%
